# Automated Web Scanner

Educational one-URL checker for HTTP security headers and basic CORS response behavior.

**Authorization:** Run only against a website you own or have explicit permission to assess. The scanner makes one GET request, does not crawl, fuzz, exploit, or attempt login. A missing header is not automatically a vulnerability.

Install dependencies with `pip install -r requirements.txt`. Then run the code cell below and call `scan_url('https://your-authorized-domain.example')`.

In [ ]:
"""
Automated Web Scanner
Checks basic HTTP security headers and CORS response behavior for ONE
user-supplied URL. Use only on websites you own or are explicitly authorized
to assess. This is a lightweight, non-invasive checker: one GET request,
no crawling, fuzzing, authentication bypass, or exploitation.
"""
from urllib.parse import urlparse
import json
import requests

SECURITY_HEADERS = {
    "Strict-Transport-Security": "Encourages HTTPS connections.",
    "Content-Security-Policy": "Restricts permitted content sources.",
    "X-Content-Type-Options": "Helps prevent MIME-type sniffing.",
    "X-Frame-Options": "Helps reduce clickjacking risk.",
    "Referrer-Policy": "Controls referrer information.",
    "Permissions-Policy": "Controls access to browser features.",
}

TEST_ORIGIN = "https://scanner-test.invalid"

def validate_url(url):
    parsed = urlparse(url.strip())
    if parsed.scheme not in ("http", "https") or not parsed.hostname:
        raise ValueError("Enter a complete http:// or https:// URL.")
    if parsed.username or parsed.password:
        raise ValueError("Do not include credentials in the URL.")
    return url.strip()

def scan_url(url, timeout=10):
    """Make one GET request and report headers; no crawling or exploit tests."""
    url = validate_url(url)
    response = requests.get(
        url,
        headers={"User-Agent": "Educational-Security-Header-Checker/1.0",
                 "Origin": TEST_ORIGIN},
        timeout=timeout,
        allow_redirects=True,
    )
    headers = {k.lower(): v for k, v in response.headers.items()}

    header_results = {}
    for header, explanation in SECURITY_HEADERS.items():
        value = headers.get(header.lower())
        header_results[header] = {
            "status": "Present" if value else "Missing",
            "value": value or "",
            "purpose": explanation,
        }

    allow_origin = headers.get("access-control-allow-origin")
    allow_credentials = headers.get("access-control-allow-credentials", "").lower()
    if not allow_origin:
        cors = "No Access-Control-Allow-Origin header was returned."
    elif allow_origin == "*":
        cors = "Wildcard origin is allowed. Review whether this is intended."
    elif allow_origin == TEST_ORIGIN:
        cors = "The server returned the supplied test Origin (possible origin reflection). Review the CORS policy."
    else:
        cors = f"The server allows this origin value: {allow_origin}"
    if allow_origin == "*" and allow_credentials == "true":
        cors += " WARNING: wildcard origin with credentials is an unsafe/invalid combination in browsers."

    return {
        "requested_url": url,
        "final_url": response.url,
        "status_code": response.status_code,
        "headers": header_results,
        "cors": {
            "allow_origin": allow_origin or "",
            "allow_credentials": allow_credentials or "",
            "assessment": cors,
        },
        "disclaimer": "A missing header is not automatically a confirmed vulnerability. Interpret results in application context.",
    }

def print_report(report):
    print(f"\nTarget: {report['requested_url']}")
    print(f"Final URL: {report['final_url']}")
    print(f"HTTP status: {report['status_code']}\n")
    print("SECURITY HEADERS")
    for name, item in report["headers"].items():
        print(f"- {name}: {item['status']}" +
              (f" ({item['value']})" if item["value"] else ""))
    print("\nCORS")
    print(report["cors"]["assessment"])
    print("\nNote:", report["disclaimer"])

if __name__ == "__main__":
    print("AUTOMATED WEB SCANNER — authorized-use educational tool")
    target = input("Enter one website URL you own or are authorized to test: ").strip()
    confirmation = input('Type AUTHORIZED to confirm permission: ').strip()
    if confirmation != "AUTHORIZED":
        raise SystemExit("Permission not confirmed. Scan cancelled.")
    try:
        result = scan_url(target)
        print_report(result)
        with open("scan_report.json", "w", encoding="utf-8") as f:
            json.dump(result, f, indent=2)
        print("\nSaved results to scan_report.json")
    except (ValueError, requests.RequestException) as exc:
        raise SystemExit(f"Scan could not be completed: {exc}")
